# Nested Square Pattern — CUDA GPU vs CPU
**Roll:** 25201328 | **Hard (CO4) – DS11** | **Concurrent Programming Practical Exam**

> Make sure: `Runtime` → `Change runtime type` → **T4 GPU** → Save

In [ ]:
# Step 1: Install & verify GPU
!pip install numba matplotlib Pillow --quiet
!nvidia-smi

In [ ]:
# Step 2: Imports
import numpy as np
import time
import math
import matplotlib.pyplot as plt
from PIL import Image
from numba import cuda

print('All imports successful')

In [ ]:
# Step 3: Configuration
WIDTH   = 4096   # image width in pixels
HEIGHT  = 4096   # image height in pixels
GAP     = 20     # spacing between squares in pixels
BLOCK   = 16     # 16x16 = 256 threads per block

print(f'Grid: {WIDTH} x {HEIGHT}')
print(f'Total pixels (= CUDA threads): {WIDTH * HEIGHT:,}')
print(f'Concentric squares: ~{(WIDTH // 2) // GAP}')

In [ ]:
# Step 4: CUDA Kernel
# Algorithm: each thread handles one pixel (x, y)
# Chebyshev distance from centre: d = max(|cx|, |cy|)
# If d % GAP == 0 -> on a square boundary -> white (255)
# Otherwise -> black (0)

@cuda.jit
def nested_squares_kernel(img, width, height, gap):
    x, y = cuda.grid(2)                      # 2D thread index
    if x >= width or y >= height:
        return                                # boundary guard
    cx   = x - width  // 2                   # centre-relative x
    cy   = y - height // 2                   # centre-relative y
    dist = max(abs(cx), abs(cy))             # Chebyshev distance
    img[y, x] = 255 if (dist % gap == 0) else 0

print('CUDA kernel compiled')

In [ ]:
# Step 5: CPU Reference (sequential)
def nested_squares_cpu(width, height, gap):
    img = np.zeros((height, width), dtype=np.uint8)
    for y in range(height):
        for x in range(width):
            cx   = x - width  // 2
            cy   = y - height // 2
            dist = max(abs(cx), abs(cy))
            if dist % gap == 0:
                img[y, x] = 255
    return img

print('CPU function defined')

In [ ]:
# Step 6: GPU Execution
h_img = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
d_img = cuda.to_device(h_img)

threads_per_block = (BLOCK, BLOCK)
bx = math.ceil(WIDTH  / BLOCK)
by = math.ceil(HEIGHT / BLOCK)
blocks_per_grid = (bx, by)

# Warm-up (absorbs JIT compilation time)
nested_squares_kernel[blocks_per_grid, threads_per_block](d_img, WIDTH, HEIGHT, GAP)
cuda.synchronize()
print('Warm-up done')

# Timed run
t0 = time.perf_counter()
nested_squares_kernel[blocks_per_grid, threads_per_block](d_img, WIDTH, HEIGHT, GAP)
cuda.synchronize()
gpu_ms = (time.perf_counter() - t0) * 1000

gpu_img = d_img.copy_to_host()
print(f'[GPU] Kernel time      : {gpu_ms:.4f} ms')
print(f'[GPU] Blocks launched  : {bx} x {by} = {bx*by:,}')
print(f'[GPU] Threads per block: {BLOCK} x {BLOCK} = {BLOCK*BLOCK}')
print(f'[GPU] Total threads    : {bx*by*BLOCK*BLOCK:,}')

In [ ]:
# Step 7: CPU Execution (sequential)
print('[CPU] Running... (this takes ~60-150 seconds)')
t0 = time.perf_counter()
cpu_img = nested_squares_cpu(WIDTH, HEIGHT, GAP)
cpu_ms = (time.perf_counter() - t0) * 1000
print(f'[CPU] Sequential time  : {cpu_ms:.4f} ms')

In [ ]:
# Step 8: Verify & Speedup
match = np.array_equal(gpu_img, cpu_img)
speedup = cpu_ms / gpu_ms

print('=' * 45)
print(f'[VERIFY]  GPU == CPU : {"PASS" if match else "FAIL"}')
print(f'[GPU]     {gpu_ms:.2f} ms')
print(f'[CPU]     {cpu_ms:.2f} ms')
print(f'[SPEEDUP] {speedup:.2f}x')
print('=' * 45)

In [ ]:
# Step 9: Save images
Image.fromarray(gpu_img).save('gpu_nested_squares.png')
Image.fromarray(cpu_img).save('cpu_nested_squares.png')
print('Images saved: gpu_nested_squares.png, cpu_nested_squares.png')

In [ ]:
# Step 10: Visualise
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(gpu_img, cmap='gray', interpolation='nearest')
axes[0].set_title(f'GPU Output\n{gpu_ms:.2f} ms', fontsize=13, fontweight='bold', color='green')
axes[0].axis('off')

axes[1].imshow(cpu_img, cmap='gray', interpolation='nearest')
axes[1].set_title(f'CPU Output\n{cpu_ms:.2f} ms', fontsize=13, fontweight='bold', color='red')
axes[1].axis('off')

bars = axes[2].bar(['GPU', 'CPU'], [gpu_ms, cpu_ms],
                   color=['#1D9E75', '#D85A30'], width=0.4, edgecolor='none')
for bar, val in zip(bars, [gpu_ms, cpu_ms]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.2f} ms', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Time (ms)', fontsize=12)
axes[2].set_title(f'Performance Comparison\nSpeedup: {speedup:.1f}x', fontsize=13, fontweight='bold')
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.suptitle(f'Nested Square Pattern  |  {WIDTH}x{HEIGHT}  |  Gap={GAP}px  |  Roll: 25201328',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('All done!')